In [ ]:
import torch
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
if not hasattr(np, 'float'):
    np.float = float # this is due to the tracker , monkey patching ok

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else :
    device = "cpu"
    dtype = torch.float32



print("Using device:", device)
print("Using dtype : " , dtype)

Using device: cpu
Using dtype :  torch.float32


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00


In [ ]:
import os
import cv2
from ultralytics import YOLO
model = YOLO('yolo11s.pt') # Using the latest 2026 model

# Tracker Part

In [ ]:
!git clone https://github.com/ifzhang/ByteTrack.git
# Replace deprecated np.float with float after cloning
%cd ByteTrack
!pip install -r requirements.txt
!python setup.py develop
!pip install cython
!pip install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip install cython_bbox
!pip install onnxruntime
!pip install loguru
!pip install lap

Cloning into 'ByteTrack'...
remote: Enumerating objects: 2007, done.
remote: Counting objects: 100% (346/346), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 2007 (delta 304), reused 296 (delta 296), pack-reused 1661 (from 1)
Receiving objects: 100% (2007/2007), 79.59 MiB | 45.53 MiB/s, done.
Resolving deltas: 100% (1160/1160), done.
/content/ByteTrack
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 41.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement onnxruntime==1.8.0 (from versions: 1.17.0, 1.17.1, 1.17.3, 1.18.0, 1.18.1, 1.19.0, 1.19.2, 1.20.0, 1.20.1, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.23.0, 1.23.1, 1.23.2, 1.24.0.dev20251031

In [ ]:
%cd "/content/ByteTrack"

/content/ByteTrack


In [ ]:
from yolox.tracker.byte_tracker import BYTETracker  # From ifzhang/ByteTrack
from yolox.tracking_utils.timer import Timer

In [ ]:
# argument class for the bytetrack
class ByteTrackArgs:
    track_thresh = 0.5
    track_buffer = 30
    match_thresh = 0.8
    aspect_ratio_thresh = 10.0
    min_box_area = 1.0
    mot20 = False

tracker = BYTETracker(ByteTrackArgs)


In [ ]:
def draw_boxes_ids(frame, x, y, w, h, track_id=None):
    x, y, w, h = map(int, [x, y, w, h])

    cv2.rectangle(
        frame,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        2
    )

    if track_id is not None:
        label = f"Person ID: {track_id}"
        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            2
        )

    return frame


# HEATMAP WITHOUT THE TILING

In [ ]:
%%writefile app.py
from fastapi import FastAPI, UploadFile, BackgroundTasks, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

import torch
import cv2
import numpy as np
import os
from ultralytics import YOLO
from yolox.tracker.byte_tracker import BYTETracker

# Monkey patch for ByteTrack compatibility with modern NumPy
if not hasattr(np, 'float'):
    np.float = float

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["Content-Disposition"],
)

UPLOAD_DIR = "uploads"
OUTPUT_DIR = "outputs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

class ByteTrackArgs:
    track_thresh = 0.5
    track_buffer = 30
    match_thresh = 0.8
    aspect_ratio_thresh = 10.0
    min_box_area = 1.0
    mot20 = False

tracker = BYTETracker(ByteTrackArgs)
model = YOLO('yolo11s.pt')

def draw_boxes_ids(frame, x, y, w, h, track_id=None):
    x, y, w, h = map(int, [x, y, w, h])
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    if track_id is not None:
        cv2.putText(frame, f"ID: {track_id}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    return frame

def process_video(input_path: str, output_path: str):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened(): return

    cap_fps = cap.get(cv2.CAP_PROP_FPS)
    TARGET_W, TARGET_H = 1280, 720
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, cap_fps, (TARGET_W, TARGET_H))

    try:
        heatmap = np.zeros((TARGET_H, TARGET_W), dtype=np.float32)
        while True:
            ret, frame_bgr = cap.read()
            if not ret: break

            frame_low = cv2.resize(frame_bgr, (TARGET_W, TARGET_H))
            img_rgb = cv2.cvtColor(frame_low, cv2.COLOR_BGR2RGB)
            results = model.predict(img_rgb, classes=[0], conf=0.2, verbose=False)

            detections = []
            for r in results:
                for box in r.boxes:
                    # Fix: Ensure correct tensor extraction
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    conf = float(box.conf[0])
                    cls_id = int(box.cls[0])
                    detections.append([x1, y1, x2, y2, conf, cls_id])

            if detections:
                dets_tensor = torch.tensor([d[:5] for d in detections])
                online_targets = tracker.update(dets_tensor.cpu(), (TARGET_H, TARGET_W), (TARGET_H, TARGET_W))

                for t in online_targets:
                    tx, ty, tw, th = map(int, t.tlwh)
                    cx, cy = tx + tw // 2, ty + th
                    if 0 <= cx < TARGET_W and 0 <= cy < TARGET_H:
                        heatmap[max(0, cy-5):min(TARGET_H, cy+6),
                                max(0, cx-5):min(TARGET_W, cx+6)] += 1.0
                    frame_low = draw_boxes_ids(frame_low, tx, ty, tw, th, t.track_id)

            heatmap *= 0.998
            h_blur = cv2.GaussianBlur(heatmap, (0, 0), sigmaX=10, sigmaY=10)
            h_norm = cv2.normalize(h_blur, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            h_color = cv2.applyColorMap(h_norm, cv2.COLORMAP_JET)
            overlay = cv2.addWeighted(frame_low, 0.6, h_color, 0.4, 0)
            out.write(overlay)
    finally:
        cap.release()
        out.release()
    open(output_path + ".done", "w").close()

@app.post("/processvideo/")
async def process_given_video(file: UploadFile = File(...), background_tasks: BackgroundTasks = BackgroundTasks()):
    input_path = os.path.join(UPLOAD_DIR, file.filename)
    output_path = os.path.join(OUTPUT_DIR, f"processed_{file.filename}")

    # Fix: Correctly read from UploadFile and write to local buffer
    with open(input_path, "wb") as f:
        content = await file.read()
        f.write(content)

    # Fix: Use .add_task() method
    background_tasks.add_task(process_video, input_path, output_path)

    return {
        "output_file": output_path,
        "status": "Processing started in background"
    }


@app.get("/download/{filename}")
async def download_fun(filename: str):

    file_path = os.path.join(OUTPUT_DIR, filename)
    done_file = file_path + ".done"

    if os.path.exists(done_file):

        return FileResponse(
            path=file_path,
            media_type="video/mp4",
            filename=filename
        )

    return {"status": "processing"}

Overwriting app.py


In [ ]:
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import subprocess

subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"])

<Popen: returncode: None args: ['uvicorn', 'app:app', '--host', '0.0.0.0', '...>

In [ ]:
!pip install fastapi uvicorn pyngrok

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
token =  userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(token)

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://laraine-unmenstruating-jean.ngrok-free.dev" -> "http://localhost:8000"
